# RealSaS Knight — Geppetto Stage27 FIRST-PASS Run-All

Demo-only FIT1 witness. Stops on the first check where all four frozen diffusion seeds satisfy the unchanged structural/metric qualification gates. Emits live check/seed logs. Product authority and generalization remain forbidden.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import gzip, hashlib, json, os, shutil, subprocess, sys, zipfile

HANDOFF = Path('/content/drive/MyDrive/RealSaS_SUBJECT2_KNIGHT_DEMO_V2_20260924/GEPPETTO_STAGE27_HANDOFF')
DRIVE_OUTPUT = Path('/content/drive/MyDrive/RealSaS_SUBJECT2_KNIGHT_DEMO_V2_20260924/GEPPETTO_STAGE27_OUTPUT')
WORK = Path('/content/realsas_geppetto_stage27')
SRC = WORK / 'source'
LOCAL_OUTPUT = WORK / 'output'

def sha256(path: Path) -> str:
    h=hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda:f.read(1<<20), b''):
            h.update(chunk)
    return h.hexdigest()

manifest_path = HANDOFF / 'HANDOFF_MANIFEST.json'
assert manifest_path.is_file(), f'MISSING_HANDOFF_MANIFEST:{manifest_path}'
manifest=json.loads(manifest_path.read_text(encoding='utf-8'))
assert manifest['schema']=='RealSaS.KnightGeppettoStage27Handoff.v1', manifest
assert manifest['status']=='PASS', manifest
assert manifest['execution_class']=='DEMO_WITNESS', manifest
assert manifest['product_authority_claimed'] is False, manifest
for rel, meta in manifest['files'].items():
    p=HANDOFF/rel
    assert p.is_file(), f'HANDOFF_FILE_MISSING:{rel}'
    assert p.stat().st_size==int(meta['bytes']), f'HANDOFF_SIZE_DRIFT:{rel}'
    got=sha256(p)
    assert got==meta['sha256'], f'HANDOFF_SHA_DRIFT:{rel}:{got}!={meta["sha256"]}'
print('GEPPETTO_STAGE27_HANDOFF_VERIFIED', manifest['execution_repo_commit'])


In [ ]:
if WORK.exists(): shutil.rmtree(WORK)
SRC.mkdir(parents=True)
LOCAL_OUTPUT.mkdir(parents=True)
MATERIALIZED = WORK / 'materialized'
MATERIALIZED.mkdir(parents=True)

compressed = dict(manifest.get('compressed_payloads') or {})
surface_transport = compressed.get('qualified_rigging_surface.json')
if surface_transport:
    assert surface_transport.get('compression') == 'gzip', surface_transport
    archive_rel = str(surface_transport['archive'])
    archive_path = HANDOFF / archive_rel
    assert archive_path.is_file(), f'COMPRESSED_SURFACE_MISSING:{archive_rel}'
    SURFACE_JSON = MATERIALIZED / 'qualified_rigging_surface.json'
    with gzip.open(archive_path, 'rb') as src, SURFACE_JSON.open('wb') as dst:
        shutil.copyfileobj(src, dst, length=1<<20)
    assert SURFACE_JSON.stat().st_size == int(surface_transport['uncompressed_bytes']), 'SURFACE_UNCOMPRESSED_SIZE_DRIFT'
    assert sha256(SURFACE_JSON) == str(surface_transport['uncompressed_sha256']), 'SURFACE_UNCOMPRESSED_SHA_DRIFT'
    print('SURFACE_TRANSPORT_GZIP_VERIFIED', archive_rel, sha256(SURFACE_JSON))
else:
    SURFACE_JSON = HANDOFF / 'qualified_rigging_surface.json'
    assert SURFACE_JSON.is_file(), f'SURFACE_JSON_MISSING:{SURFACE_JSON}'

archive=HANDOFF/'realsas_execution_source.zip'
with zipfile.ZipFile(archive,'r') as zf:
    zf.extractall(SRC)
assert (SRC/'tools/training/run_knight_geppetto_demo_fit_v2_first_pass.py').is_file()
subprocess.run([sys.executable,'-m','pip','install','--no-input','--progress-bar','off','-r',str(SRC/'requirements/mainline-ci.txt')],check=True)
print('SOURCE_ARCHIVE_READY', sha256(archive))


In [ ]:
cmd=[
    sys.executable, str(SRC/'tools/training/run_knight_geppetto_demo_fit_v2_first_pass.py'),
    '--surface-json', str(SURFACE_JSON),
    '--surface-qualification-json', str(HANDOFF/'rigging_surface_qualification.json'),
    '--teacher-target-npz', str(HANDOFF/'teacher_target.npz'),
    '--teacher-target-report', str(HANDOFF/'teacher_target_report.json'),
    '--preregistration-ir', str(HANDOFF/'model_fit_preregistration.json'),
    '--experiment-prereg', str(HANDOFF/'experiment_preregistration.json'),
    '--model-source', str(HANDOFF/'model_source.py'),
    '--camera-set-json', str(HANDOFF/'qualified_camera_set.json'),
    '--observation-dir', str(HANDOFF/'observations'),
    '--output-dir', str(LOCAL_OUTPUT),
]
env=dict(os.environ)
env['PYTHONPATH']=str(SRC)+((':'+env['PYTHONPATH']) if env.get('PYTHONPATH') else '')
subprocess.run(cmd,cwd=SRC,env=env,check=True)


In [ ]:
out_manifest_path=LOCAL_OUTPUT/'GEPPETTO_KNIGHT_OUTPUT_MANIFEST.json'
assert out_manifest_path.is_file(), 'OUTPUT_MANIFEST_MISSING'
out_manifest=json.loads(out_manifest_path.read_text(encoding='utf-8'))
assert out_manifest['schema']=='RealSaS.KnightGeppettoFitOutputManifest.v1', out_manifest
assert out_manifest['status']=='PASS', out_manifest
assert out_manifest['product_authority_claimed'] is False, out_manifest
for role, meta in out_manifest['files'].items():
    p=LOCAL_OUTPUT/meta['path']
    assert p.is_file(), f'OUTPUT_FILE_MISSING:{role}:{p}'
    assert sha256(p)==meta['sha256'], f'OUTPUT_SHA_DRIFT:{role}'
result=json.loads((LOCAL_OUTPUT/'GEPPETTO_KNIGHT_RESULT.json').read_text(encoding='utf-8'))
assert result['all_diffusion_seeds_passed'] is True, result
assert result['teacher_feedback_during_free_running_inference'] is False, result
assert result['teacher_inference_inputs_used'] is False, result
assert result['historical_checkpoint_loaded'] is False, result
runtime=dict(result.get('runtime_environment') or {})
assert runtime.get('cuda_available') is True, runtime
assert runtime.get('torch_version'), runtime
assert runtime.get('torch_cuda_runtime'), runtime
assert (runtime.get('gpu0') or {}).get('name'), runtime
assert result['product_authority_claimed'] is False, result

if DRIVE_OUTPUT.exists(): shutil.rmtree(DRIVE_OUTPUT)
shutil.copytree(LOCAL_OUTPUT, DRIVE_OUTPUT)
marker={
    'schema':'RealSaS.KnightGeppettoStage27RunAllComplete.v1',
    'status':'PASS',
    'handoff_manifest_sha256':sha256(manifest_path),
    'output_manifest_sha256':sha256(DRIVE_OUTPUT/'GEPPETTO_KNIGHT_OUTPUT_MANIFEST.json'),
    'product_authority_claimed':False,
    'generalization_claimed':False,
}
(DRIVE_OUTPUT/'RUN_ALL_COMPLETE.json').write_text(json.dumps(marker,indent=2,sort_keys=True)+'\n',encoding='utf-8')
print('GEPPETTO_STAGE27_RUN_ALL_PASS')
print(json.dumps(marker,sort_keys=True))
